In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
pd.set_option('display.width', 120)


In [4]:
df = pd.read_csv('nykaa_campaign_data_og.csv')
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['Month'] = df['Date'].dt.to_period('M').astype(str)
df['CTR_%'] = (df['Clicks'] / df['Impressions'] * 100).round(2)
df['Lead_Rate_%'] = (df['Leads'] / df['Clicks'] * 100).round(2)
df['Conversion_Rate_%'] = (df['Conversions'] / df['Leads'] * 100).round(2)
df['Overall_Conv_%'] = (df['Conversions'] / df['Impressions'] * 100).round(4)
df['Revenue_per_Conversion'] = (df['Revenue'] / df['Conversions']).round(2)
df.to_csv('nykaa_cleaned.csv', index=False)
print(df.shape)

(55555, 22)


# ROI by Campaign Type

In [5]:
q1 = df.groupby('Campaign_Type').agg(
    Avg_ROI=('ROI', 'mean'),
    Total_Revenue=('Revenue', 'sum'),
    Total_Spend=('Acquisition_Cost', 'sum'),
    Avg_Conversion_Rate=('Conversion_Rate_%', 'mean'),
    Campaigns=('Campaign_ID', 'count')
).round(2).sort_values('Avg_ROI', ascending=False)
print(q1)

               Avg_ROI  Total_Revenue  Total_Spend  Avg_Conversion_Rate  Campaigns
Campaign_Type                                                                     
Social Media      2.75     5751837620   4148449.24                54.89      11114
Paid Ads          2.72     5751468983   4214354.89                54.91      11116
SEO               2.71     5698831847   4196047.50                55.06      11079
Influencer        2.70     5769064044   4180655.95                55.02      11134
Email             2.68     5685161788   4224008.78                55.18      11112


# ROI by individual Channel

In [6]:
df_channel = df.assign(Channel_Used=df['Channel_Used'].str.split(', ')).explode('Channel_Used')
q1b = df_channel.groupby('Channel_Used').agg(
    Avg_ROI=('ROI', 'mean'),
    Total_Revenue=('Revenue', 'sum'),
    Campaigns=('Campaign_ID', 'count')
).round(2).sort_values('Avg_ROI', ascending=False)
print(q1b)

              Avg_ROI  Total_Revenue  Campaigns
Channel_Used                                   
Instagram        2.74     9564113010      18493
Email            2.73     9519798466      18461
YouTube          2.72     9573677663      18525
Google           2.71     9581611838      18553
Facebook         2.70     9445844772      18344
WhatsApp         2.70     9557967220      18527


# Q1 chart

In [7]:
fig, ax = plt.subplots(figsize=(8,5))
q1['Avg_ROI'].sort_values().plot(kind='barh', ax=ax, color='#e91e8c')
ax.set_title('Average ROI by Campaign Type')
plt.tight_layout()
plt.savefig('q1_roi_by_campaign_type.png', dpi=120)
plt.close()
print("Chart saved!")

Chart saved!


# Acquisition Cost vs ROI

In [8]:
corr = df['Acquisition_Cost'].corr(df['ROI'])
print(f"Correlation: {corr:.3f}")

cost_75 = df['Acquisition_Cost'].quantile(0.75)
roi_25 = df['ROI'].quantile(0.25)
inefficient = df[(df['Acquisition_Cost'] >= cost_75) & (df['ROI'] <= roi_25)]
print(f"Inefficient campaigns: {len(inefficient)} ({len(inefficient)/len(df)*100:.1f}%)")
print(inefficient.groupby('Campaign_Type').size().sort_values(ascending=False))

Correlation: -0.371
Inefficient campaigns: 11301 (20.3%)
Campaign_Type
Email           2313
Paid Ads        2292
Influencer      2235
SEO             2234
Social Media    2227
dtype: int64


# Q2 chart

In [9]:
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(df['Acquisition_Cost'], df['ROI'], alpha=0.15, s=8, color='#7b2d8e')
ax.set_xlabel('Acquisition Cost')
ax.set_ylabel('ROI')
ax.set_title('Acquisition Cost vs ROI')
plt.tight_layout()
plt.savefig('q2_cost_vs_roi.png', dpi=120)
plt.close()


Chart saved!


# Performance by Target Audience

In [11]:
q3 = df.groupby('Target_Audience').agg(
    Total_Revenue=('Revenue', 'sum'),
    Avg_ROI=('ROI', 'mean'),
    Avg_Conversion_Rate=('Conversion_Rate_%', 'mean'),
    Avg_Engagement=('Engagement_Score', 'mean'),
    Campaigns=('Campaign_ID', 'count')
).round(2).sort_values('Total_Revenue', ascending=False)
print(q3)


                       Total_Revenue  Avg_ROI  Avg_Conversion_Rate  Avg_Engagement  Campaigns
Target_Audience                                                                              
Premium Shoppers          5852284945     2.80                55.09           13.83      11182
College Students          5786247365     2.74                55.12           13.85      11078
Working Women             5743709729     2.67                55.02           13.81      11224
Tier 2 City Customers     5701063841     2.68                54.92           13.76      11100
Youth                     5573058402     2.68                54.90           13.67      10971


# Q3 Chart

In [12]:
fig, ax = plt.subplots(figsize=(8,5))
q3['Total_Revenue'].sort_values().plot(kind='barh', ax=ax, color='#00abc7')
ax.set_title('Total Revenue by Target Audience')
plt.tight_layout()
plt.savefig('q3_revenue_by_audience.png', dpi=120)
plt.close()
print("Chart saved!")

Chart saved!


# Funnel Analysis

In [13]:
funnel = {
    'Impressions': df['Impressions'].sum(),
    'Clicks': df['Clicks'].sum(),
    'Leads': df['Leads'].sum(),
    'Conversions': df['Conversions'].sum()
}
prev = None
for stage, val in funnel.items():
    if prev:
        drop = (1 - val/prev) * 100
        print(f"{stage}: {val:,}  (drop: {drop:.1f}%)")
    else:
        print(f"{stage}: {val:,}")
    prev = val

Impressions: 3,060,407,471
Clicks: 260,445,757  (drop: 91.5%)
Leads: 104,291,797  (drop: 60.0%)
Conversions: 57,380,922  (drop: 45.0%)


# Q4 Chart 

In [14]:
fig, ax = plt.subplots(figsize=(7,5))
ax.bar(list(funnel.keys()), list(funnel.values()), color=['#e91e8c','#7b2d8e','#00abc7','#2ecc71'])
ax.set_title('Overall Marketing Funnel')
plt.tight_layout()
plt.savefig('q4_funnel.png', dpi=120)
plt.close()
print("Chart saved!")

Chart saved!


In [15]:
q5 = df.groupby(['Campaign_Type','Language','Target_Audience']).agg(
    Avg_ROI=('ROI','mean'),
    Campaigns=('Campaign_ID','count'),
    Total_Revenue=('Revenue','sum')
).round(2)
q5 = q5[q5['Campaigns'] >= 30].sort_values('Avg_ROI', ascending=False)
print(q5.head(10))

                                         Avg_ROI  Campaigns  Total_Revenue
Campaign_Type Language Target_Audience                                    
Influencer    Hindi    Premium Shoppers     3.16        543      301492750
Email         Tamil    Working Women        3.08        553      305824810
Paid Ads      Hindi    Youth                3.05        533      290928566
Social Media  Hindi    College Students     3.04        533      301330070
Influencer    Bengali  College Students     3.02        529      291895914
SEO           English  College Students     3.02        564      323619771
Social Media  Tamil    College Students     3.02        551      306822251
              English  Youth                3.00        540      284535689
Paid Ads      English  College Students     2.99        537      283921603
SEO           Hindi    Working Women        2.98        532      268525209


In [1]:
exit()